In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [3]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [10]:

tablaCurrency = "Currency"
esquemaSales = "Sales"

dimensionCurrency = pd.read_sql_table(tablaCurrency, motorBaseDatos, esquemaSales)
dimensionCurrency
# dimensionCurrency.head()
# dimensionCurrency.info()
# dimensionCurrency.Name.unique()

c:\Users\dange.DANGERPC\OneDrive\Escritorio\etl-aventure-works\my_env\Lib\site-packages\pandas\io\sql.py:1737: SAWarning: Did not recognize type 'Name' of column 'Name'
  self.meta.reflect(bind=self.con, only=[table_name], views=True)


,CurrencyCode,Name,ModifiedDate
0,AED,Emirati Dirham,2008-04-30
1,AFA,Afghani,2008-04-30
2,ALL,Lek,2008-04-30
3,AMD,Armenian Dram,2008-04-30
4,ANG,Netherlands Antillian Guilder,2008-04-30
...,...,...,...
100,VEB,Bolivar,2008-04-30
101,VND,Dong,2008-04-30
102,XOF,CFA Franc BCEAO,2008-04-30
103,ZAR,Rand,2008-04-30


TRANSFORMACION

In [9]:

dimensionCurrency.rename(columns={
    'Name' : 'CurrencyName',
    'CurrencyCode' : 'CurrencyAlternateKey'
    }, inplace=True)


dimensionCurrency.drop('ModifiedDate', axis=1, inplace=True)



dimensionCurrency
# dimensionCurrency.info()


,CurrencyAlternateKey,CurrencyName
0,AED,Emirati Dirham
1,AFA,Afghani
2,ALL,Lek
3,AMD,Armenian Dram
4,ANG,Netherlands Antillian Guilder
...,...,...
100,VEB,Bolivar
101,VND,Dong
102,XOF,CFA Franc BCEAO
103,ZAR,Rand


CARGAR A LA BODEGA

In [5]:
dimensionCurrency.to_sql('dimensionCurrency',motorBodegaDatos, if_exists='replace',index_label='CurrencyKey')

105